In [1]:
import torch
import torch.nn as nn
import numpy as np 
import matplotlib.pyplot as plt
import sys 
import os
import glob

In [2]:
sys.path.append("/media/ana-caznok/SSD-08/recon-segment/")
sys.path.append("/media/ana-caznok/SSD-08/recon-segment/transforms")
sys.path.append("/media/ana-caznok/SSD-08/recon-segment/configs")
sys.path.append("/media/ana-caznok/SSD-08/recon-segment/plots")

In [3]:
from seg_recon_vit3d_overlap import *
from utils.read_yaml import read_yaml
from utils.model_select import model_select

In [4]:
base_path = "/media/ana-caznok/SSD-08/recon-segment/"
config = read_yaml(base_path + 'configs/test_overlap.yaml')
model,load = model_select(config)

RecDecoder(
  (decoder): Sequential(
    (0): ConvTranspose2d(768, 384, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1))
    (1): BatchNorm2d(384, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): ConvTranspose2d(384, 192, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1))
    (4): BatchNorm2d(192, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): ReLU(inplace=True)
    (6): ConvTranspose2d(192, 96, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1))
    (7): BatchNorm2d(96, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (8): ReLU(inplace=True)
    (9): ConvTranspose2d(96, 61, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1))
    (10): BatchNorm2d(61, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (11): ReLU(inplace=True)
    (12): Conv2d(61, 61, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  )
)
Looking for checkpoint in models/test_overlap.pth. Exact path only: 

In [27]:
def get_output_shape(transform_class, transform_params, input_shape):
    """
    Computes the output shape of a PyTorch nn transform given the transform class, 
    its parameters, and the input tensor shape.

    Parameters:
    - transform_class (torch.nn.Module): The class of the PyTorch transform (e.g., nn.Conv2d).
    - transform_params (dict): Parameters required to instantiate the transform.
    - input_shape (tuple): The shape of the input tensor (e.g., (1, 3, 224, 224)).

    Returns:
    - tuple: Output tensor shape after applying the transform.
    """

    # Instantiate the transform using the provided class and parameters
    transform = transform_class(**transform_params)

    # Create a dummy input tensor with the specified input shape
    dummy_input = torch.randn(*input_shape)

    # Apply the transform to the dummy input without tracking gradients
    with torch.no_grad():
        output = transform(dummy_input)

    # Return the output shape as a tuple
    return tuple(output.shape)


In [8]:
model.encoder

Encoder(
  (embedding_function): Embedd(
    (patch_embedding_func): Conv2d(4, 768, kernel_size=(32, 32), stride=(16, 16))
  )
  (attention): ViT_Attention(
    (q_proj): Linear(in_features=768, out_features=768, bias=True)
    (k_proj): Linear(in_features=768, out_features=768, bias=True)
    (v_proj): Linear(in_features=768, out_features=768, bias=True)
    (out_proj): Linear(in_features=768, out_features=768, bias=True)
  )
)

In [45]:
B, C, Y, X = 1, 4, 256, 256
E = 768
img = torch.randn(*(B,C,Y,X))

In [46]:
x,h,w = model.encoder(img)

In [47]:
b, t, e = x.shape
decoder_input = x.transpose(1, 2).contiguous().view(B, E, h, w)
b_d, e_d, h, w = decoder_input.shape
decoder_input_shape = (b_d,e_d,h, w)

In [38]:
decoder_input_shape

(1, 768, 15, 15)

In [33]:
x.transpose(1, 2).contiguous().view(B, E, h, w).shape

torch.Size([1, 768, 15, 15])

In [ ]:
for i in range(len(channels) - 1):
            layers.append(nn.ConvTranspose2d(
                in_channels=channels[i],
                out_channels=channels[i+1],
                kernel_size=4,
                stride=2,
                padding=1
            ))

In [5]:
upsample_factor = model.decoder.upsample_factor
upsample_layers = model.decoder.num_upsample_layers 
channels =  model.decoder.dec_channels

In [136]:
decoder_input_shape = (b_d,e_d,h, w)
conv_shapes = [decoder_input_shape]
for i in range(len(channels)-1): 
  conv_output_shape = get_output_shape(
          nn.ConvTranspose2d,
          {"in_channels": channels[i],
            "out_channels": channels[i+1], 
            "kernel_size": 5,
            "stride": 2,
            "padding": 1},
          decoder_input_shape
      )
  conv_shapes.append(conv_output_shape)
  decoder_input_shape = conv_output_shape
#print("Output shape:", conv_output_shape)

In [137]:
conv_output_shape = get_output_shape(
                    nn.Conv2d,
                    {"in_channels": channels[-1],
                        "out_channels": 61, 
                        "kernel_size": 2,
                        "padding": 1},
                    decoder_input_shape) 
conv_shapes.append(conv_output_shape)

In [126]:
#kernel size: ou 5 ou 6

In [138]:
conv_shapes

[(1, 768, 15, 15),
 (1, 384, 31, 31),
 (1, 192, 63, 63),
 (1, 96, 127, 127),
 (1, 61, 255, 255),
 (1, 61, 256, 256)]

In [76]:
256 + 256/2

384.0